In [ ]:
import pandas as pd
import numpy as np

# 1. Veriyi Yükleme
df = pd.read_csv('../scraping/data/cleaned_data.csv')

# ---------------------------------------------------------
# AŞAMA 1: KAYIP VERİ ANALİZİ VE DOLDURMA (IMPUTATION)
# ---------------------------------------------------------
print("--- Eksik Veri Analizi (İşlem Öncesi) ---")
eksik_veriler = df.isnull().sum()
print(eksik_veriler[eksik_veriler > 0])

def kayip_verileri_doldur(dataframe):
    df_temp = dataframe.copy()
    
    # Sayısal sütunlardaki eksiklikleri Medyan (Ortanca) ile dolduruyoruz.
    # Medyan, aykırı değerlerden (çok büyük/küçük uçuk rakamlar) ortalamaya göre daha az etkilenir.
    sayisal_sutunlar = df_temp.select_dtypes(include=['float64', 'int64']).columns
    for sutun in sayisal_sutunlar:
        if df_temp[sutun].isnull().sum() > 0:
            df_temp[sutun] = df_temp[sutun].fillna(df_temp[sutun].median())
            
    # Kategorik sütunlardaki (metin içeren) eksiklikleri Mod (En çok tekrar eden değer) ile dolduruyoruz.
    kategorik_sutunlar = df_temp.select_dtypes(include=['object']).columns
    for sutun in kategorik_sutunlar:
        if df_temp[sutun].isnull().sum() > 0:
            df_temp[sutun] = df_temp[sutun].fillna(df_temp[sutun].mode()[0])
            
    return df_temp

df = kayip_verileri_doldur(df)

print("\n--- Eksik Veri Analizi (İşlem Sonrası) ---")
print("Kalan eksik veri sayısı:", df.isnull().sum().sum())

--- Eksik Veri Analizi (İşlem Öncesi) ---
bina_yasi         1
isitma            1
banyo_sayisi    155
ilce              4
mahalle           4
dtype: int64

--- Eksik Veri Analizi (İşlem Sonrası) ---
Kalan eksik veri sayısı: 0


C:\Users\Baran\AppData\Local\Temp\ipykernel_18808\4127361811.py:25: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  kategorik_sutunlar = df_temp.select_dtypes(include=['object']).columns


In [ ]:
# ---------------------------------------------------------
# AŞAMA 2: ÖZNİTELİK ÇIKARIMI (FEATURE ENGINEERING)
# ---------------------------------------------------------
print("\n--- Öznitelik Çıkarımı Yapılıyor ---")

# 1. Kat Oranı (Bulunduğu Kat / Toplam Kat Sayısı)
# Evin binanın neresinde olduğunu oranlar (Örn: 0.0 giriş, 1.0 en üst kat, 0.5 orta kat)
# Not: Hatalı veri olup kat_sayisi'nin 0 olma ihtimaline karşı np.where ile sıfıra bölme hatasını engelliyoruz.
df['kat_orani'] = np.where(df['kat_sayisi'] > 0, df['kat'] / df['kat_sayisi'], 0)

# 2. Metrekare Farkı (Kayıp Alan)
# Brüt metrekare ile net metrekare arasındaki farkı bularak "kullanılamayan" alanı tespit ediyoruz.
df['metrekare_farki'] = df['metrekare_brut'] - df['metrekare_net']

# 3. Metrekare Verimliliği (Net m2 / Brüt m2)
# Evin yüzde kaçının net kullanım alanı olduğunu gösteren bir oran (Örn: %85 verimli)
df['metrekare_verimliligi'] = np.where(df['metrekare_brut'] > 0, df['metrekare_net'] / df['metrekare_brut'], 0)

# 4. Oda - Salon Ayrımı (İsteğe bağlı, kategorik dönüşüm değil metin parçalama işlemidir)
# "3+1" gibi veriyi "3" ve "1" olarak ayırıp toplam oda sayısını bulmak ANN için yararlıdır.
if 'oda_salon' in df.columns:
    df[['oda_sayisi', 'salon_sayisi']] = df['oda_salon'].str.split('+', expand=True)
    df['oda_sayisi'] = pd.to_numeric(df['oda_sayisi'], errors='coerce').fillna(1)
    df['salon_sayisi'] = pd.to_numeric(df['salon_sayisi'], errors='coerce').fillna(0)
    df['toplam_oda'] = df['oda_sayisi'] + df['salon_sayisi']
    df.drop('oda_salon', axis=1, inplace=True) # Eski metin halini siliyoruz


print("\nYeni oluşturulan matematiksel özniteliklerden ilk 5 satır:")
print(df[['kat', 'kat_sayisi', 'kat_orani', 'metrekare_brut', 'metrekare_net', 'metrekare_farki', 'metrekare_verimliligi']].head())


--- Öznitelik Çıkarımı Yapılıyor ---

Yeni oluşturulan matematiksel özniteliklerden ilk 5 satır:
   kat  kat_sayisi  kat_orani  metrekare_brut  metrekare_net  metrekare_farki  \
0  2.0         4.0       0.50            80.0            1.0             79.0   
1  3.0         4.0       0.75           120.0          105.0             15.0   
2  2.0         5.0       0.40           120.0          100.0             20.0   
3  1.0         4.0       0.25           120.0          115.0              5.0   
4  2.0         4.0       0.50           120.0          115.0              5.0   

   metrekare_verimliligi  
0               0.012500  
1               0.875000  
2               0.833333  
3               0.958333  
4               0.958333  


In [ ]:
# Metrekare farkı 50'den büyük olan kayıtları filtreleme
hatali_metrekare_verileri = df[df['metrekare_farki'] > 50]

# Bu kayıtların sayısını bulup yazdırma
silinecek_kayit_sayisi = len(hatali_metrekare_verileri)
print(f"Metrekare farkı 50'den büyük olan hatalı kayıt sayısı: {silinecek_kayit_sayisi}")

Metrekare farkı 50'den büyük olan hatalı kayıt sayısı: 155


In [ ]:
# Sadece metrekare farkı 50'ye eşit veya küçük olan (mantıklı) verileri tutuyoruz
df = df[df['metrekare_farki'] <= 50]

# İsteğe bağlı: İşlem sonrası kaç satır kaldığını görmek için
print(f"Hatalı kayıtlar silindi. Güncel veri seti boyutu: {df.shape}")

Hatalı kayıtlar silindi. Güncel veri seti boyutu: (13627, 16)


In [ ]:
# Veriyi aynı isimle CSV olarak kaydetme
# index=False parametresi, Pandas'ın satır numaralarını (0, 1, 2...) yeni bir sütun olarak kaydetmesini engeller
df.to_csv('../scraping/data/cleaned_data.csv', index=False)

print("Veri temizlendi ve başarıyla 'cleaned_data.csv' dosyasının üzerine kaydedildi.")

Veri temizlendi ve başarıyla 'cleaned_data.csv' dosyasının üzerine kaydedildi.
